In [ ]:
"""
=============================================================================
  BiLSTM PIPELINE — Multivariate Time-Series Classification (4 Classes)
=============================================================================

Dataset layout (after inspection):
  X  : (N, 10, 24)  — 10 timesteps × 24 pre-normalised features
  y  : (N,)         — integer labels 0-3
  masks: (N, 10)    — boolean validity mask (all True in training split)

Key data facts that drove design choices
  • Class imbalance  : [6358, 6776, 5830, 3025]  → class weights + focal loss
  • Static features  : cols 4-11 have near-zero temporal variance → separated
  • Dynamic features : cols 12-23 carry most of the temporal signal
  • Redundant pair   : feat 16 ≡ feat 19 (r = 1.0)            → dropped feat 19
  • Near-redundant   : feat 2 ≈ feat 14 (r = 0.99)            → kept, handled by
                                                                  L2 regularisation

Pipeline steps
  1. Load & audit data
  2. Feature engineering (temporal + interaction)
  3. Class-weight computation
  4. Model build (BiLSTM + Attention + residual)
  5. Callbacks (LR-schedule, early-stop, checkpoint)
  6. Train
  7. Evaluate (accuracy, F1, confusion matrix)
=============================================================================
"""

In [1]:
# ─── 0. Imports ──────────────────────────────────────────────────────────────
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"          # suppress TF info spam
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
# ─── 1. LOAD & AUDIT DATA ────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading data")
print("=" * 60)

DATA_DIR = "/mnt/user-data/uploads"

X_train    = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/X_train.npy")         # (21989, 10, 24)
X_val      = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/X_val.npy")           # (2761,  10, 24)
X_test     = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/X_test.npy")          # (2750,  10, 24)
y_train    = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/y_train.npy")         # (21989,)
y_val      = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/y_val.npy")           # (2761,)
y_test     = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/y_test.npy")          # (2750,)
masks_train = np.load("/Users/amitsingh/ML_Projects/WarfareAI/datasets/LSTM/masks_train.npy")    # (21989, 10) bool

N_TIMESTEPS  = X_train.shape[1]   # 10
N_FEATURES   = X_train.shape[2]   # 24
N_CLASSES    = len(np.unique(y_train))  # 4

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}    |  y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print(f"Classes : {N_CLASSES}  |  Timesteps : {N_TIMESTEPS}  |  Features : {N_FEATURES}")
print(f"Class counts (train): {np.bincount(y_train)}")

STEP 1 — Loading data
X_train : (21989, 10, 24)  |  y_train : (21989,)
X_val   : (2761, 10, 24)    |  y_val   : (2761,)
X_test  : (2750, 10, 24)   |  y_test  : (2750,)
Classes : 4  |  Timesteps : 10  |  Features : 24
Class counts (train): [6358 6776 5830 3025]


In [5]:
# ─── 2. FEATURE ENGINEERING ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Feature engineering")
print("=" * 60)

def engineer_features(X: np.ndarray) -> np.ndarray:
    """
    Takes raw X of shape (N, T, 24) and returns an enriched array (N, T, F_new).

    Engineering catalogue
    ─────────────────────
    [A] Drop feat 19   — perfectly correlated with feat 16 (r = 1.0); adds zero
                         information and wastes model capacity.

    [B] Temporal delta — Δx_t = x_t − x_{t-1} for the 12 dynamic features
                         (cols 12-23). Captures rate-of-change which is a strong
                         discriminative signal in time-series problems. t=0 gets Δ=0.

    [C] Rolling mean   — mean of x_{t-2..t} for dynamic features. Smooths
                         short-term noise and gives the model a local trend.

    [D] Rolling std    — std of x_{t-2..t} for dynamic features. Encodes local
                         volatility; useful to separate classes with similar mean
                         but different variability.

    [E] Feature norms  — L2 norm across *all* features at each timestep.
                         Gives the model a single-number summary of overall
                         activity at that moment.

    [F] Static-feature interaction — sum of the static cols 4-11 grouped as
                         a scalar per timestep (same value at every t, but
                         explicit interaction helps dense layers).

    Total output width = 23 (orig – 1 drop) + 12 (Δ) + 12 (roll-mean)
                       + 12 (roll-std) + 1 (norm) + 1 (static-sum) = 61
    """
    N, T, _ = X.shape

    # [A] Drop feature 19 (duplicate of 16)
    DROP_COLS = [19]
    keep_mask = [i for i in range(24) if i not in DROP_COLS]
    X_base = X[:, :, keep_mask]                              # (N, T, 23)

    # Identify dynamic feature indices *in the original array* (cols 12-23)
    DYN_ORIG = list(range(12, 24))
    DYN_ORIG = [c for c in DYN_ORIG if c not in DROP_COLS]  # remove 19 → 11 cols
    dyn_idx  = [keep_mask.index(c) for c in DYN_ORIG]       # positions in X_base

    X_dyn = X_base[:, :, dyn_idx]                           # (N, T, 11)

    # [B] Temporal first-difference (Δ)
    delta = np.zeros_like(X_dyn)
    delta[:, 1:, :] = X_dyn[:, 1:, :] - X_dyn[:, :-1, :]  # (N, T, 11)

    # [C] Rolling mean over a window of 3 (t-2, t-1, t)
    roll_mean = np.zeros_like(X_dyn)
    for t in range(T):
        start = max(0, t - 2)
        roll_mean[:, t, :] = X_dyn[:, start:t+1, :].mean(axis=1)

    # [D] Rolling std over a window of 3
    roll_std = np.zeros_like(X_dyn)
    for t in range(T):
        start = max(0, t - 2)
        window = X_dyn[:, start:t+1, :]
        # std = 0 when window size is 1 (no variance); that's fine
        roll_std[:, t, :] = window.std(axis=1)

    # [E] L2 norm across all original features at each timestep
    norm_feat = np.linalg.norm(X_base, axis=-1, keepdims=True)  # (N, T, 1)

    # [F] Static feature sum (cols 4-11 in original → cols 4-11 in X_base)
    STATIC_ORIG = list(range(4, 12))
    stat_idx = [keep_mask.index(c) for c in STATIC_ORIG]
    static_sum = X_base[:, :, stat_idx].sum(axis=-1, keepdims=True)  # (N, T, 1)

    # Concatenate everything
    X_eng = np.concatenate(
        [X_base, delta, roll_mean, roll_std, norm_feat, static_sum], axis=-1
    )
    return X_eng.astype(np.float32)


X_train_eng = engineer_features(X_train)
X_val_eng   = engineer_features(X_val)
X_test_eng  = engineer_features(X_test)

F_ENG = X_train_eng.shape[-1]
print(f"Original feature width : {N_FEATURES}")
print(f"Engineered feature width: {F_ENG}")
print(f"  = 23 (base) + 11 (Δ) + 11 (roll-mean) + 11 (roll-std) + 1 (norm) + 1 (static-sum)")



STEP 2 — Feature engineering
Original feature width : 24
Engineered feature width: 58
  = 23 (base) + 11 (Δ) + 11 (roll-mean) + 11 (roll-std) + 1 (norm) + 1 (static-sum)


In [6]:
# ─── 3. CLASS WEIGHTS ────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Class-weight computation")
print("=" * 60)

# sklearn balanced weights: w_c = N / (n_classes * n_c)
cw_vals = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(N_CLASSES),
    y=y_train
)
class_weight_dict = {i: float(w) for i, w in enumerate(cw_vals)}
print(f"Class weights : {class_weight_dict}")



STEP 3 — Class-weight computation
Class weights : {0: 0.8646193771626297, 1: 0.8112824675324676, 2: 0.9429245283018868, 3: 1.8172727272727274}


In [8]:
# ─── 4. MODEL ARCHITECTURE ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Building BiLSTM model")
print("=" * 60)

# ── 4a. Temporal Attention Layer ─────────────────────────────────────────────
class TemporalAttention(layers.Layer):
    """
    Additive (Bahdanau-style) self-attention over the time axis.

    Each timestep produces a scalar score; softmax normalises across T;
    the output is the weighted sum of hidden states.

    Why: with only 10 timesteps the sequence is short but attention still
    helps the model focus on the most discriminative moments rather than
    treating every step equally.
    """
    def __init__(self, units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=False)
        self.v = layers.Dense(1, use_bias=False)

    def call(self, h):  # h : (B, T, H)
        score = self.v(tf.nn.tanh(self.W(h)))   # (B, T, 1)
        alpha = tf.nn.softmax(score, axis=1)     # (B, T, 1)
        context = tf.reduce_sum(alpha * h, axis=1)  # (B, H)
        return context, tf.squeeze(alpha, -1)    # also return weights for viz


def build_bilstm_model(
    timesteps: int,
    features: int,
    n_classes: int,
    lstm_units_1: int = 128,
    lstm_units_2: int = 64,
    dense_units: int = 128,
    dropout_rate: float = 0.40,
    l2: float = 1e-4,
) -> keras.Model:
    """
    Architecture overview
    ─────────────────────
    Input (T, F)
      → Gaussian Noise (augmentation, σ=0.01)          [reg: data augmentation]
      → BiLSTM-1 (128 × 2 = 256 output, return_seq=True) [captures long-range deps]
      → Dropout + LayerNorm
      → BiLSTM-2 (64  × 2 = 128 output, return_seq=True) [refine representation]
      → Dropout + LayerNorm
      → TemporalAttention (64 units)                    [focus on key timesteps]
      → Residual branch: GlobalAvgPool on BiLSTM-2 output
        ↕  (concatenated with attention context)         [residual keeps local info]
      → Dense-128 (GELU) + Dropout + BatchNorm
      → Dense-64  (GELU) + Dropout
      → Dense-n_classes (softmax)

    Regularisation strategy
      • L2 on all LSTM kernels and Dense layers
      • Spatial dropout on LSTM outputs (preserves temporal correlations)
      • Gaussian input noise (makes model robust to measurement noise)
      • Label smoothing in loss (helps with class overlap)
    """
    reg = regularizers.l2(l2)
    inp = keras.Input(shape=(timesteps, features), name="input")

    # Data-augmentation noise (only active during training)
    x = layers.GaussianNoise(stddev=0.01, name="input_noise")(inp)

    # ── BiLSTM block 1 ──────────────────────────────────────────────────────
    x = layers.Bidirectional(
        layers.LSTM(
            lstm_units_1,
            return_sequences=True,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            dropout=0.10,               # input-gate dropout
            recurrent_dropout=0.00,     # keep recurrent path stable
        ),
        name="bilstm_1",
    )(x)
    x = layers.SpatialDropout1D(rate=dropout_rate * 0.5, name="sdrop_1")(x)
    x = layers.LayerNormalization(name="ln_1")(x)

    # ── BiLSTM block 2 ──────────────────────────────────────────────────────
    h = layers.Bidirectional(
        layers.LSTM(
            lstm_units_2,
            return_sequences=True,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            dropout=0.10,
        ),
        name="bilstm_2",
    )(x)
    h = layers.SpatialDropout1D(rate=dropout_rate * 0.5, name="sdrop_2")(h)
    h = layers.LayerNormalization(name="ln_2")(h)

    # ── Temporal Attention ──────────────────────────────────────────────────
    ctx, _attn_weights = TemporalAttention(units=64, name="temporal_attn")(h)

    # ── Residual global-average branch ─────────────────────────────────────
    gap = layers.GlobalAveragePooling1D(name="gap")(h)   # (B, 128)
    merged = layers.Concatenate(name="merge")([ctx, gap]) # (B, 256)

    # ── Classification head ─────────────────────────────────────────────────
    z = layers.Dense(dense_units, activation="gelu",
                     kernel_regularizer=reg, name="dense_1")(merged)
    z = layers.Dropout(dropout_rate, name="drop_3")(z)
    z = layers.BatchNormalization(name="bn_1")(z)

    z = layers.Dense(dense_units // 2, activation="gelu",
                     kernel_regularizer=reg, name="dense_2")(z)
    z = layers.Dropout(dropout_rate * 0.5, name="drop_4")(z)

    out = layers.Dense(n_classes, activation="softmax",
                       name="output")(z)

    model = keras.Model(inputs=inp, outputs=out, name="BiLSTM_Attention")
    return model


model = build_bilstm_model(
    timesteps=N_TIMESTEPS,
    features=F_ENG,
    n_classes=N_CLASSES,
)
model.summary()


STEP 4 — Building BiLSTM model


Model: "BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 10, 58)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_noise         │ (None, 10, 58)    │          0 │ input[0][0]       │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 10, 256)   │    191,488 │ input_noise[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sdrop_1             │ (None, 10, 256)   │          0 │ bilstm_1[0][0]    │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ln_1                │ (None, 10, 256)   │        512 │ sdrop_1[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 10, 128)   │    164,352 │ ln_1[0][0]        │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sdrop_2             │ (None, 10, 128)   │          0 │ bilstm_2[0][0]    │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ln_2                │ (None, 10, 128)   │        256 │ sdrop_2[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attn       │ [(None, 128),     │      8,256 │ ln_2[0][0]        │
│ (TemporalAttention) │ (None, 10)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 128)       │          0 │ ln_2[0][0]        │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merge (Concatenate) │ (None, 256)       │          0 │ temporal_attn[0]… │
│                     │                   │            │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ merge[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_3 (Dropout)    │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 128)       │        512 │ drop_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_4 (Dropout)    │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 4)         │        260 │ drop_4[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 406,788 (1.55 MB)

 Trainable params: 406,532 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [9]:
# ─── 5. COMPILE ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Compiling")
print("=" * 60)

# Label smoothing: prevents overconfident predictions and acts as regulariser.
# α=0.1 → true-class target becomes 0.925 instead of 1.0
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.10)

# AdamW: Adam + decoupled weight decay.  Better generalisation than plain Adam.
optimizer = keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.AUC(multi_label=False, name="auc"),
    ],
)

# One-hot encode labels for CategoricalCrossentropy
y_train_oh = keras.utils.to_categorical(y_train, N_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   N_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  N_CLASSES)


STEP 5 — Compiling


In [29]:
# ─── 7. TRAIN ────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Training")
print("=" * 60)

import os
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Callbacks (SAFE — no TensorBoard)
cb_list = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    ModelCheckpoint(
        filepath="./checkpoints/best_model.keras",   # modern format
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

BATCH_SIZE = 256
EPOCHS = 100

history = model.fit(
    X_train_eng,
    y_train_oh,
    validation_data=(X_val_eng, y_val_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=1
)


STEP 7 — Training
Epoch 1/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - acc: 0.5190 - auc: 0.7915 - loss: 1.2952
Epoch 1: val_loss improved from None to 0.95988, saving model to ./checkpoints/best_model.keras
86/86 ━━━━━━━━━━━━━━━━━━━━ 46s 445ms/step - acc: 0.6189 - auc: 0.8749 - loss: 1.1131 - val_acc: 0.6758 - val_auc: 0.9302 - val_loss: 0.9599 - learning_rate: 0.0010
Epoch 2/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step - acc: 0.7226 - auc: 0.9304 - loss: 0.9358
Epoch 2: val_loss improved from 0.95988 to 0.86455, saving model to ./checkpoints/best_model.keras
86/86 ━━━━━━━━━━━━━━━━━━━━ 37s 428ms/step - acc: 0.7306 - auc: 0.9349 - loss: 0.9163 - val_acc: 0.7095 - val_auc: 0.9431 - val_loss: 0.8645 - learning_rate: 0.0010
Epoch 3/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step - acc: 0.7530 - auc: 0.9449 - loss: 0.8720
Epoch 3: val_loss improved from 0.86455 to 0.82792, saving model to ./checkpoints/best_model.keras
86/86 ━━━━━━━━━━━━━━━━━━━━ 36s 419ms/step - acc: 0.7562 - auc: 0.9471 -

In [32]:
# ─── 8. EVALUATE ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8 — Evaluation")
print("=" * 60)

# Load best checkpoint
best_model = keras.models.load_model(
    "./checkpoints/best_model.keras",
    custom_objects={"TemporalAttention": TemporalAttention},
)

# ── 8a. Test-set metrics ─────────────────────────────────────────────────────
test_loss, test_acc, test_auc = best_model.evaluate(
    X_test_eng, y_test_oh, batch_size=BATCH_SIZE, verbose=0
)
print(f"\nTest  loss : {test_loss:.4f}")
print(f"Test  acc  : {test_acc:.4f}")
print(f"Test  AUC  : {test_auc:.4f}")

# ── 8b. Per-class F1 ─────────────────────────────────────────────────────────
y_pred_proba = best_model.predict(X_test_eng, batch_size=BATCH_SIZE, verbose=0)
y_pred       = y_pred_proba.argmax(axis=1)

print("\n--- Classification Report ---")
print(classification_report(
    y_test, y_pred,
    target_names=[f"Class {i}" for i in range(N_CLASSES)],
    digits=4,
))

# ── 8c. Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Confusion matrix
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[f"Pred {i}" for i in range(N_CLASSES)],
    yticklabels=[f"True {i}" for i in range(N_CLASSES)],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (Test Set)", fontsize=14)

# Training curves
epochs_done = range(1, len(history.history["acc"]) + 1)
axes[1].plot(epochs_done, history.history["acc"],    label="Train Acc",  lw=2)
axes[1].plot(epochs_done, history.history["val_acc"], label="Val Acc",   lw=2, ls="--")
axes[1].plot(epochs_done, history.history["loss"],   label="Train Loss", lw=2, alpha=0.7)
axes[1].plot(epochs_done, history.history["val_loss"], label="Val Loss", lw=2, ls="--", alpha=0.7)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Metric value")
axes[1].set_title("Training Curves", fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.savefig("./results.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved confusion matrix + training curves → ./results.png")

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)


STEP 8 — Evaluation

Test  loss : 0.7463
Test  acc  : 0.7811
Test  AUC  : 0.9631

--- Classification Report ---
              precision    recall  f1-score   support

     Class 0     0.8084    0.7083    0.7550       792
     Class 1     0.7556    0.8430    0.7969       847
     Class 2     0.8361    0.8361    0.8361       726
     Class 3     0.6909    0.6909    0.6909       385

    accuracy                         0.7811      2750
   macro avg     0.7727    0.7696    0.7697      2750
weighted avg     0.7830    0.7811    0.7803      2750


Saved confusion matrix + training curves → ./results.png

PIPELINE COMPLETE
